In [ ]:
import yaml
import sys
sys.path.append('..')
from src.data_loader import load_tcga_expression, convert_ensembl_to_symbols
from src.preprocessing import apply_log2_transformation, filter_low_expression_genes, prepare_train_test_split
from src.feature_selection import run_differential_expression, balance_classes_smote, custom_boruta
from src.models import get_benchmark_models, get_optimized_rf
from src.evaluation import evaluate_test_set
from src.visualization import plot_comprehensive_dashboard

In [ ]:
with open('../config/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
df = load_tcga_expression(f"../{cfg['data']['raw_path']}")
df = convert_ensembl_to_symbols(df)
df = apply_log2_transformation(df, threshold=cfg['preprocessing']['log2_threshold'])
df_filtered = filter_low_expression_genes(df, threshold=cfg['preprocessing']['expression_threshold'])

In [ ]:
X_train, X_test, y_train, y_test = prepare_train_test_split(df_filtered)
de_genes, _ = run_differential_expression(X_train, y_train)
X_train_balanced, y_train_balanced = balance_classes_smote(X_train[de_genes], y_train)
selected_genes, _ = custom_boruta(X_train_balanced, y_train_balanced, trials=50)

In [ ]:
models = get_benchmark_models()
results = {}
for name, model in models.items():
    model.fit(X_train_balanced[selected_genes], y_train_balanced)
    metrics, _, _ = evaluate_test_set(model, X_test[selected_genes], y_test)
    results[name] = metrics
display(pd.DataFrame(results).T.round(4).sort_values("AUC", ascending=False))

In [ ]:
rf_opt = get_optimized_rf()
rf_opt.fit(X_train_balanced[selected_genes], y_train_balanced)
_, y_pred, y_proba = evaluate_test_set(rf_opt, X_test[selected_genes], y_test)
importances = pd.Series(rf_opt.feature_importances_, index=selected_genes).sort_values(ascending=False)
plot_comprehensive_dashboard(y_test, y_pred, y_proba, importances, pd.DataFrame(results).T)